SEQUENCE TO SEQUENCE

Le reti viste finora (RNN, LSTM, GRU) hanno in comune di ricevere una sequenza e di produrre un output
Esemipio:
input: questo file è bellissimo
output: sentiment=positivo
Questa architettura è chiamata Many-to-One
Moti input, un solo output
Ma esistono problemi completamente diversi
Esempio:
input: traduci 'Io mangio una mela'
output: 'I eat an apple'
Qui abbiamo più parole in ingresso e più parole in uscita.
Non possiamo usare una LSTM, serve qualcosa di diverso
Nasce così Sequence-to-Sequence

Definizione
Sequence-to-Sequence è una rete neurale che trasforma una sequenza in un'altra sequenza
Testo -> Testo
Audio -> Testo
Testo -> Codice
Testo -> Riassunto
L'importante è che sia input sia output siano delle sequenze.

La Seq2Seq è stasta utilizzata in moltissimo problemi
Traduzione automatica, Chatbot,  Riassunto automatico, Correzione grammaticale, Specch Recognition, Generazione di codice

La Seq2Seq divide il lavoro, una rete legge una rete scrive
Frase -> Encoder -> Significato -> Decoder -> Nuova frase.
Digerisce un'intera sequenza prima di produrne una nuova.
L'input può avere una lunghezza n e l'output avere una lunghezza m totalmente indipendenti

L'Encoder
l'encoder legge  una parola alla volta, alla fine otteniamo  uno stato finale che rappresenta, idealmente, il significato dell'intera frase (contiene il significato in forma numerica).
E' un ascoltatore attento, ascolta elemento per elemento (uno alla volta) e aggiorna il suo stato interno per riassumere l'informazione

Il Context Vector
Lo stato finale viene chiamato Context Vector.
la frase 'io mangio la mela' potrebbe diventare [0.42,-1.38,0.91,...]
Non sappiamo cosa significano quei numeri, sappiamo che quel vettore rappresenta la frase 'io mangio la mela'

Il Decoder
Il Decoder riceve solo il Context Vector, da li deve ricostruire tutta la frase
Il Decoder è spesso un'altra LSTM
Il Decoder funziona in modo autoregressivo
Il Decoder nè come uno scrittore, non inizia a scrivere fino a quando l'Encoder non ha finito, basandosi sull'ultimo stato dell'encoder
Genera una parola, poi usa la parola per produrre la successiva
I token <Start> (SOS) e <End> (EOS) indicano rispettivamente l'inizio e la fine della sequenza. Fino a quando non genera <End>, continua a generare parole.

Come viene addestrato?
supponiamo che la frase di input sia 'io mangio una mela' il decoder produce 'I eat a orange' confrontando le due frasi si notano le differenze che generano un errore.
L'errore torna indietro tramite la Backpropagation Through Time (BPTT)
I pesi vengono midifcati
Ripetendo questo processo milioni di volte, il modello impara.

L'Encoder comprime l'informazione in uno strato nascosto di lunghezza variabili, che rappresenta il significato semantico della frase
Il Decoder lavora un passo alla volta: l'output prodotto al tempo 't' diventa input per il tempo t+1, permettendo alla rete di mantenere coerenza sintattica.
Lo stato finale della cella RNN dell'Encoder funge da stato iniziale per la cella del Decoder, trasferendo la memoria tra le due reti.
Il Decoder è autoregressivo, usa quello che ha appena scritto per decidere cosa scrivere dopo

Il vetore di contesto è l'unico collegamento fisico tra le due reti (Encoder Decoder). E' un vettore di numeri a virgola mobile che deve contenere ogni dettaglio necessario alla traduzione.
Senza questo veettore, il Decoder non avrebbe alcuna informazione su ciò che l'Encoder ha processato, rendendo impossibile la generazione di un output coerente.

Il grande limite del Seq2Seq classico
Immagina una frase molto lunga, l'encoder deve comprimere tutto in un unico Context Vector, è come cercare di riassumere un libro in un post-it
Per frasi brevi funziona
Per frasi lunghe perde informazioni
- Collo di bottiglia: se una frase è lunga 128 caratteri il Seq2Seq deve comprimere l'informazione in un vettore di lunghezza fissa, il che può causare perdita di dettagli
- Spazio Latente: il vettore vive in uno spazio multidimensionale dove frasi con significati simili sono posizionate vicino tra loro
- Stato di Pensiero: spesso descritto come una rappresentazione astratta del concetto, indipendentemente dalla lingua specifica originale.
L'inizializzazione del Decoder: il veettore di contesto viene tipicamente usato per impostare i valori iniziali di 'h' e 'c' nelle cella LSTM e GRU del Decoder.

Man mano che le sequenze di allungano, il vettore di contesto fatica, è come cercare di riassumere i promessi sposti in un unico post.
La trama principale resta, ma le sfumature spariscono.
La capcità dipende dalla dimensione n, più grande è n, più dettagli memoriziamo.

Nasce l'Attention
Per risolvere questo problema viene introdotto il meccanismo di Attention
L'idea è semplice:
Invece di ricordare tutto il decoder 'guarda' ogni volta la parte della frase che gli serve.
Non usa più solo Context Vector, ma tutti gli stati prodotti dall'encoder
In questo modo, quando deve tradurre 'mela', presta maggior attenzione alla parola ('mela') invece di basarsi esclusivamente su una memoria compressa.
Questa innovazione migliora significativamente le prestazioni su sequenze lunghe.

Dal Seq2Seq ai Transformer
Il Seq2Seq è quindi un passaggio fondamentale nella storia delle reti
RNN -> LSTM -> GRU -> Sequence-to-Sequence -> Attention -> Transformer -> GPT, Llama, Qwen, Claude, Gemini
L'attention elimina il collo di bottiglia del Context Vector, mentre i Transformer abbandonano del tutto RNN e LSTM, basandosi esclusivamente sul meccanismo di Attention

In [1]:
import os
import requests  # NECESSARIO: gestisce la richiesta HTTP al server
import zipfile   # NECESSARIO: estrae il file .txt dallo zip
import io        # NECESSARIO: gestisce i dati scaricati in memoria

# 1. CONFIGURAZIONE BACKEND
# Nel 2026, Keras 3 permette di scegliere il motore di calcolo. 
# Usiamo PyTorch per la sua efficienza e diffusione professionale.
os.environ["KERAS_BACKEND"] = "torch"

import numpy as np
import keras
from keras import layers
import re

# ---------------------------------------------------------
# 2. PREPARAZIONE DEI DATI (Dataset Reale Eng-Ita)
# ---------------------------------------------------------

# Scarichiamo un file zip contenente migliaia di coppie di frasi tradotte
url = "http://www.manythings.org/anki/ita-eng.zip"
data_dir = "datasets" # Cartella locale dove salvare i dati

if not os.path.exists(data_dir):
    os.makedirs(data_dir)

path_to_file = os.path.join(data_dir, "ita.txt")

# Scarichiamo manualmente se il file non esiste
if not os.path.exists(path_to_file):
    print("Scaricamento dataset in corso...")
    # Definiamo un User-Agent per ingannare il server
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    r = requests.get(url, headers=headers)
    
    # Estraiamo il contenuto dello zip direttamente in memoria
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall(data_dir)
    print("Download completato.")

def preprocess_sentence(s):
    """Pulisce il testo e aggiunge i token di controllo necessari al Decoder."""
    s = s.lower().strip()
    # Inserisce uno spazio tra parole e punteggiatura (es: "casa?" -> "casa ?")
    s = re.sub(r"([?.!,])", r" \1 ", s)
    # [start] dice al decoder di iniziare a scrivere, [end] di fermarsi
    return f"[start] {s.strip()} [end]"

num_samples = 15000  # Limitiatamo a 15k frasi per un addestramento rapido (pochi minuti)
input_texts = []     # Conterrà le frasi in inglese
target_texts = []    # Conterrà le frasi in italiano

with open(path_to_file, encoding="utf-8") as f:
    lines = f.read().split("\n")
    for line in lines[:num_samples]:
        # Il file è tab-separated: "English \t Italian \t Attribution"
        parts = line.split("\t")
        if len(parts) >= 2:
            input_texts.append(parts[0].lower())
            target_texts.append(preprocess_sentence(parts[1]))

# ---------------------------------------------------------
# 3. VETTORIZZAZIONE (Text to Numbers)
# ---------------------------------------------------------

vocab_size = 5000  # Consideriamo solo le 5000 parole più frequenti
seq_len = 15       # Lunghezza massima delle frasi (tronca o riempie di zeri)

# Layer per trasformare l'inglese in numeri (Encoder)
source_vectorization = layers.TextVectorization(
    max_tokens=vocab_size, 
    output_sequence_length=seq_len
)
# Layer per trasformare l'italiano in numeri (Decoder)
target_vectorization = layers.TextVectorization(
    max_tokens=vocab_size, 
    output_sequence_length=seq_len + 1 # +1 per gestire lo shift del Teacher Forcing
)

# "Adattiamo" i layer ai nostri testi per costruire il vocabolario
source_vectorization.adapt(input_texts)
target_vectorization.adapt(target_texts)

# ---------------------------------------------------------
# 4. DEFINIZIONE ARCHITETTURA (Encoder-Decoder)
# ---------------------------------------------------------

latent_dim = 256  # Dimensione dello "spazio del pensiero" (vettore di contesto)

# --- ENCODER (L'Interprete) ---
# Riceve la sequenza inglese e ne estrae il significato astratto
encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="encoder_input")
x = layers.Embedding(vocab_size, 128)(encoder_inputs) # Trasforma ID in vettori densi
# return_state=True ci permette di ottenere gli stati interni h e c (Context Vector)
_, state_h, state_c = layers.LSTM(latent_dim, return_state=True)(x)
encoder_states = [state_h, state_c] # Questo è il ponte verso il decoder

# --- DECODER (Lo Scrittore) ---
# Riceve la frase italiana (parziale) e gli stati dell'encoder
decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="decoder_input")
x_dec = layers.Embedding(vocab_size, 128)(decoder_inputs)
# Inizializziamo la LSTM del decoder con gli stati "pensati" dall'encoder
decoder_lstm = layers.LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(x_dec, initial_state=encoder_states)
# Layer finale: per ogni posizione, decide quale parola del vocabolario usare
decoder_dense = layers.Dense(vocab_size, activation="softmax")
output = decoder_dense(decoder_outputs)

# Creazione del modello completo
model = keras.Model([encoder_inputs, decoder_inputs], output)

# Compilazione: usiamo sparse_categorical_crossentropy perché i target sono numeri interi
model.compile(
    optimizer="rmsprop", 
    loss="sparse_categorical_crossentropy", 
    metrics=["accuracy"]
)

# ---------------------------------------------------------
# 5. TRAINING (Logica Teacher Forcing)
# ---------------------------------------------------------

# Trasformiamo i testi in matrici numeriche
X_enc = source_vectorization(input_texts)
Y_all = target_vectorization(target_texts)

# Il Decoder riceve la frase italiana senza l'ultima parola
X_dec = Y_all[:, :-1]  
# Il Target (cosa deve indovinare) è la frase italiana senza la prima parola ([start])
# In questo modo il modello impara: "se ho [start] e lo stato X, la parola successiva è Y"
Y_target = Y_all[:, 1:] 

print("Inizio addestramento...")
model.fit([X_enc, X_dec], Y_target, batch_size=64, epochs=15, validation_split=0.2)

# ---------------------------------------------------------
# 6. INFERENZA (Test con mano)
# ---------------------------------------------------------

# Prepariamo un dizionario inverso per trasformare i numeri in parole leggibili
ita_vocab = target_vectorization.get_vocabulary()
ita_index_lookup = dict(enumerate(ita_vocab))

def translate(eng_sentence):
    """Funzione per tradurre una frase nuova passo dopo passo."""
    # 1. Trasformiamo la frase inglese in input per l'encoder
    input_seq = source_vectorization([eng_sentence])
    
    # 2. Partiamo con il token iniziale [start]
    decoded_sentence = "[start]"
    
    for i in range(seq_len):
        # 3. Trasformiamo quello che abbiamo scritto finora in numeri
        tokenized_target = target_vectorization([decoded_sentence])[:, :-1]
        
        # 4. Chiediamo al modello di predire la prossima parola
        predictions = model.predict([input_seq, tokenized_target], verbose=0)
        
        # 5. Prendiamo la parola con la probabilità più alta all'ultima posizione
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = ita_index_lookup[sampled_token_index]
        
        # 6. Se il modello dice [end], la traduzione è finita
        if sampled_token == "[end]":
            break
        decoded_sentence += " " + sampled_token
        
    return decoded_sentence.replace("[start]", "").strip()

# --- TEST FINALE ---
print("\n--- TEST DI TRADUZIONE ---")
test_phrases = ["it is cold", "i am a student", "help me", "go away", "i love you"]
for p in test_phrases:
    print(f"INGLESE: {p}")
    print(f"ITALIANO: {translate(p)}\n")

Scaricamento dataset in corso...
Download completato.
Inizio addestramento...
Epoch 1/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 103s 548ms/step - accuracy: 0.7784 - loss: 1.6684 - val_accuracy: 0.7633 - val_loss: 1.5430
Epoch 2/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 113s 600ms/step - accuracy: 0.8229 - loss: 1.2047 - val_accuracy: 0.8101 - val_loss: 1.3860
Epoch 3/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 118s 627ms/step - accuracy: 0.8357 - loss: 1.1398 - val_accuracy: 0.8021 - val_loss: 1.3645
Epoch 4/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 98s 522ms/step - accuracy: 0.8391 - loss: 1.1080 - val_accuracy: 0.8131 - val_loss: 1.3329
Epoch 5/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 91s 486ms/step - accuracy: 0.8455 - loss: 1.0697 - val_accuracy: 0.8147 - val_loss: 1.3111
Epoch 6/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 110s 585ms/step - accuracy: 0.8487 - loss: 1.0279 - val_accuracy: 0.8162 - val_loss: 1.2737
Epoch 7/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 132s 704ms/step - accuracy: 0.8515 - loss: 0.9854 - val_accuracy: 0.8189 - val_loss: 1.2500
